# TP Deep Learning : bilan des codes du CM3 « Outils pour le deep learning » (pages 30 à 77)

Ce notebook reprend **tous les codes du cours à partir de la page 30**, dans l'ordre des diapositives,
avec des commentaires en français. Le numéro de page est indiqué dans chaque titre.

| Partie | Contenu | Pages |
|---|---|---|
| 1 | Graphe de calcul : NumPy, TensorFlow, PyTorch (CPU / GPU) | 30 – 35 |
| 2 | PyTorch : tenseurs, autograd, `nn`, optimiseurs, modules | 36 – 46 |
| 3 | PyTorch : exemple complet FashionMNIST | 47 – 51 |
| 4 | PyTorch : modèles pré-entraînés | 52 |
| 5 | TensorFlow 1.x vs 2.x, TensorBoard, Keras | 53 – 60 |
| 6 | Calcul distribué, PyTorch vs TensorFlow | 61 – 64 |
| 7 | TensorFlow 1.x pas à pas : de `placeholder` à `tf.layers` | 66 – 77 |

> **Installation** (une seule fois) : `pip install numpy matplotlib tensorflow tensorboard tf-keras torch torchvision`

> **Remarque sur TensorFlow 1.x.** Plusieurs codes du cours utilisent l'ancienne API (TF 1.x : `placeholder`,
> `Session`…). Elle n'existe plus directement en TensorFlow 2 ; on y accède par `tf.compat.v1`.
> Chaque code TF 1.x est placé dans un bloc `with tf.Graph().as_default():`, ce qui recrée le mode
> « graphe statique » **localement**, sans perturber les cellules TF 2.x du notebook.

---
# Partie 1 : graphe de calcul et gradients (pages 30 à 35)

On étudie le graphe suivant, où `x`, `y`, `z` sont des matrices de taille $N \times D$ :

$$
a = x \odot y, \qquad b = a + z, \qquad c = \sum_{i,j} b_{ij}
$$

($\odot$ : produit élément par élément.) Le calcul analytique donne
$\frac{\partial c}{\partial x} = y$, $\frac{\partial c}{\partial y} = x$, $\frac{\partial c}{\partial z} = \mathbf{1}$.

## 1.1 NumPy : gradients écrits à la main (page 30)

**Problèmes soulignés par le cours :** NumPy ne s'exécute pas sur GPU, et les gradients doivent être
implémentés « à la main ».

In [1]:
import numpy as np

# Graine du générateur aléatoire : on obtient toujours les mêmes nombres
np.random.seed(0)

# Dimensions des matrices : N lignes, D colonnes
N, D = 3, 4

# Trois matrices N x D tirées selon une loi normale centrée réduite
x = np.random.randn(N, D)
y = np.random.randn(N, D)
z = np.random.randn(N, D)

# ---------------- Passe avant (forward) ----------------
a = x * y        # produit élément par élément
b = a + z        # somme élément par élément
c = np.sum(b)    # somme de tous les éléments -> scalaire

# ---------------- Passe arrière (backward) : règle de la chaîne ----------------
grad_c = 1.0                         # dc/dc = 1 : point de départ
grad_b = grad_c * np.ones((N, D))    # c = sum(b)  =>  dc/db = 1 partout
grad_a = grad_b.copy()               # b = a + z   =>  le gradient passe tel quel à a...
grad_z = grad_b.copy()               # ... et à z (.copy() : deux tableaux distincts en mémoire)
grad_x = grad_a * y                  # a = x * y   =>  dc/dx = dc/da * y
grad_y = grad_a * x                  #                 dc/dy = dc/da * x

print("c =", c)
print("grad_x =\n", grad_x)
print("grad_y =\n", grad_y)
print("grad_z =\n", grad_z)

# Vérification : on retrouve le résultat analytique
print(np.allclose(grad_x, y), np.allclose(grad_y, x), np.all(grad_z == 1))

c = 6.717008537800067
grad_x =
 [[ 0.76103773  0.12167502  0.44386323  0.33367433]
 [ 1.49407907 -0.20515826  0.3130677  -0.85409574]
 [-2.55298982  0.6536186   0.8644362  -0.74216502]]
grad_y =
 [[ 1.76405235  0.40015721  0.97873798  2.2408932 ]
 [ 1.86755799 -0.97727788  0.95008842 -0.15135721]
 [-0.10321885  0.4105985   0.14404357  1.45427351]]
grad_z =
 [[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
True True True


## 1.2 TensorFlow : le même graphe, gradients automatiques (pages 31 et 32)

Les étapes annotées dans le cours :

1. **Définition des variables à la base du graphe** : les `placeholder` (« trous » remplis à l'exécution) ;
2. **Création du graphe de calcul** : `a`, `b`, `c` (rien n'est encore calculé) ;
3. **Définition des gradients** : `tf.gradients` ajoute au graphe les noeuds du calcul des gradients ;
4. **Calcul de c et des gradients** : `sess.run` exécute le graphe avec les valeurs de `feed_dict`.

In [2]:
# Basic computational graph
import numpy as np
np.random.seed(0)
import tensorflow as tf
tf1 = tf.compat.v1   # accès à l'API TensorFlow 1.x

N, D = 3, 4

# Graphe statique local (style TF 1.x)
with tf.Graph().as_default():
    # 1. Variables à la base du graphe : des placeholders
    x = tf1.placeholder(tf.float32)
    y = tf1.placeholder(tf.float32)
    z = tf1.placeholder(tf.float32)

    # 2. Création du graphe de calcul (symbolique)
    a = x * y
    b = a + z
    c = tf.reduce_sum(b)

    # 3. Définition des gradients de c par rapport à x, y, z
    grad_x, grad_y, grad_z = tf1.gradients(c, [x, y, z])

    # 4. Exécution du graphe dans une session
    with tf1.Session() as sess:
        # feed_dict : valeur numérique associée à chaque placeholder
        values = {
            x: np.random.randn(N, D),
            y: np.random.randn(N, D),
            z: np.random.randn(N, D),
        }
        out = sess.run([c, grad_x, grad_y, grad_z],
                       feed_dict=values)
        c_val, grad_x_val, grad_y_val, grad_z_val = out

print("c =", c_val)
print("grad_x =\n", grad_x_val)
print("grad_y =\n", grad_y_val)
print("grad_z =\n", grad_z_val)

ModuleNotFoundError: No module named 'tensorflow'

## 1.3 TensorFlow sur GPU (page 33)

Il suffit de construire le graphe dans un bloc `with tf.device('/gpu:0'):`.
Les matrices sont plus grandes (3000 × 4000) pour que le GPU soit utile.

*Ajout :* si aucun GPU n'est disponible, on se replie sur le CPU pour que la cellule s'exécute quand même.

In [ ]:
import numpy as np
np.random.seed(0)
import tensorflow as tf
tf1 = tf.compat.v1

N, D = 3000, 4000

# '/gpu:0' = premier GPU de la machine ; repli sur le CPU s'il n'y en a pas (ajout)
device = '/gpu:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
print("Device utilisé :", device)

with tf.Graph().as_default():
    # Toutes les opérations créées dans ce bloc seront placées sur le device choisi
    with tf.device(device):
        x = tf1.placeholder(tf.float32)
        y = tf1.placeholder(tf.float32)
        z = tf1.placeholder(tf.float32)

        a = x * y
        b = a + z
        c = tf.reduce_sum(b)

    grad_x, grad_y, grad_z = tf1.gradients(c, [x, y, z])

    with tf1.Session() as sess:
        values = {
            x: np.random.randn(N, D),
            y: np.random.randn(N, D),
            z: np.random.randn(N, D),
        }
        out = sess.run([c, grad_x, grad_y, grad_z],
                       feed_dict=values)
        c_val, grad_x_val, grad_y_val, grad_z_val = out

print("c =", c_val)
print("forme de grad_x :", grad_x_val.shape)

## 1.4 PyTorch sur CPU (page 34)

- **Définition des variables à la base du graphe** : tenseurs avec `requires_grad=True` ;
- **Création du graphe de calcul** : le graphe est construit **à la volée** pendant la passe avant ;
- **Calcul des gradients** : un seul appel, `c.backward()`.

> `Variable` est obsolète depuis PyTorch 0.4 : `torch.randn(N, D, requires_grad=True)` suffit.
> Le code du cours fonctionne toujours (`Variable(...)` renvoie simplement un tenseur).

In [ ]:
import torch
from torch.autograd import Variable

N, D = 3, 4

# requires_grad=True : PyTorch enregistre les opérations sur ces tenseurs pour la dérivation
x = Variable(torch.randn(N, D),
             requires_grad=True)
y = Variable(torch.randn(N, D),
             requires_grad=True)
z = Variable(torch.randn(N, D),
             requires_grad=True)

# Création du graphe de calcul (passe avant)
a = x * y
b = a + z
c = torch.sum(b)

# Calcul des gradients : dc/dx, dc/dy, dc/dz sont rangés dans .grad
c.backward()

print(x.grad.data)   # = y
print(y.grad.data)   # = x
print(z.grad.data)   # = matrice de 1

## 1.5 PyTorch sur GPU (page 35)

Seule différence : `.cuda()` copie les tenseurs sur la carte graphique.
*Ajout :* la cellule ne s'exécute que si un GPU CUDA est disponible.

In [ ]:
import torch
from torch.autograd import Variable

if torch.cuda.is_available():
    N, D = 3, 4

    # .cuda() place le tenseur sur le GPU : tous les calculs suivants s'y feront
    x = Variable(torch.randn(N, D).cuda(),
                 requires_grad=True)
    y = Variable(torch.randn(N, D).cuda(),
                 requires_grad=True)
    z = Variable(torch.randn(N, D).cuda(),
                 requires_grad=True)

    a = x * y
    b = a + z
    c = torch.sum(b)

    c.backward()

    print(x.grad.data)
    print(y.grad.data)
    print(z.grad.data)
else:
    print("Pas de GPU CUDA disponible : cellule ignorée.")

---
# Partie 2 : PyTorch (pages 36 à 46)

## Concepts fondamentaux (page 37)

- **`torch.Tensor`** : similaire à un tableau NumPy, mais peut s'exécuter sur GPU ;
- **`torch.autograd`** : construction des graphes de calcul et calcul automatique des gradients ;
- **`torch.nn.Module`** : une couche de réseau de neurones ; stocke ses paramètres.

## Le problème traité dans toute cette partie

Un réseau à deux couches (une couche cachée ReLU), une **fonction de coût L2**, un apprentissage par
**descente du gradient**, sur des **données aléatoires** :

$$
h = x W_1, \qquad h_{\text{relu}} = \max(h, 0), \qquad \hat y = h_{\text{relu}} W_2,
\qquad L = \sum (\hat y - y)^2
$$

`N` = taille du lot, `D_in` = dimension d'entrée, `H` = neurones cachés, `D_out` = dimension de sortie.

*Ajout dans chaque boucle :* affichage de la perte toutes les 100 itérations pour voir qu'elle diminue.

## 2.1 PyTorch : tenseurs, gradients à la main (pages 38 et 39)

- **Création de tenseurs aléatoires** : données `x`, `y` et paramètres `W1`, `W2` (sur CPU **ou GPU**) ;
- **Phase de propagation** : calcul des prédictions et de la fonction de coût ;
- **Phase de rétro-propagation** : calcul **manuel** des gradients ;
- **Descente du gradient**.

In [ ]:
import torch

device = torch.device('cpu')   # ou torch.device('cuda') pour le GPU

# Création de tenseurs aléatoires : données et paramètres
N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in, device=device)
y = torch.randn(N, D_out, device=device)
w1 = torch.randn(D_in, H, device=device)
w2 = torch.randn(H, D_out, device=device)

learning_rate = 1e-6
for t in range(500):
    # ---- Phase de propagation ----
    h = x.mm(w1)                        # produit matriciel : couche cachée
    h_relu = h.clamp(min=0)             # ReLU : max(h, 0)
    y_pred = h_relu.mm(w2)              # couche de sortie
    loss = (y_pred - y).pow(2).sum()    # coût L2

    if t % 100 == 99:
        print(t, loss.item())

    # ---- Phase de rétro-propagation (calcul manuel) ----
    grad_y_pred = 2.0 * (y_pred - y)          # dL/dy_pred
    grad_w2 = h_relu.t().mm(grad_y_pred)      # dL/dW2 = h_relu^T . dL/dy_pred
    grad_h_relu = grad_y_pred.mm(w2.t())      # dL/dh_relu = dL/dy_pred . W2^T
    grad_h = grad_h_relu.clone()              # copie pour ne pas modifier grad_h_relu
    grad_h[h < 0] = 0                         # la ReLU bloque le gradient là où h < 0
    grad_w1 = x.t().mm(grad_h)                # dL/dW1 = x^T . dL/dh

    # ---- Descente du gradient ----
    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2

## 2.2 PyTorch : autograd (pages 40 et 41)

- Créer des tenseurs avec `requires_grad=True` **active autograd** ;
- les opérations sur ces tenseurs **construisent un graphe de calcul** ;
- propagation : plus besoin de conserver les valeurs intermédiaires ;
- rétro-propagation : `loss.backward()` calcule automatiquement les gradients ;
- descente du gradient : `no_grad` (pas de graphe pour cette partie) et `grad.zero_()` (ne pas accumuler les gradients).

In [ ]:
import torch

N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)
# Seuls les poids ont besoin d'un gradient
w1 = torch.randn(D_in, H, requires_grad=True)
w2 = torch.randn(H, D_out, requires_grad=True)

learning_rate = 1e-6
for t in range(500):
    # Phase de propagation : x . W1 -> ReLU -> . W2
    y_pred = x.mm(w1).clamp(min=0).mm(w2)
    loss = (y_pred - y).pow(2).sum()

    if t % 100 == 99:
        print(t, loss.item())

    # Phase de rétro-propagation automatique : remplit w1.grad et w2.grad
    loss.backward()

    # Descente du gradient, sans enregistrer ces opérations dans le graphe
    with torch.no_grad():
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad
        # Les gradients s'accumulent d'un backward() à l'autre : remise à zéro
        w1.grad.zero_()
        w2.grad.zero_()

## 2.3 PyTorch : définir ses propres fonctions autograd (page 42)

- définir la fonction de **propagation** (`forward`) ;
- définir la fonction de **rétro-propagation** (`backward`) ;
- définir une fonction « **helper** » pour s'en servir simplement ;
- `ctx` : valeur « cache » qui transmet des informations de `forward` à `backward`.

In [ ]:
import torch

class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)     # on garde x en cache pour la passe arrière
        return x.clamp(min=0)        # ReLU : max(x, 0)

    @staticmethod
    def backward(ctx, grad_y):
        # grad_y : gradient de la perte par rapport à la sortie de la ReLU
        x, = ctx.saved_tensors       # récupération de l'entrée sauvegardée
        grad_input = grad_y.clone()
        grad_input[x < 0] = 0        # dérivée de ReLU : 0 si x < 0, 1 sinon
        return grad_input            # gradient par rapport à l'entrée x

def my_relu(x):
    # fonction "helper" : .apply() exécute la Function
    return MyReLU.apply(x)

*Ajout :* utilisation de `my_relu` dans le réseau de la section 2.2.

In [ ]:
N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)
w1 = torch.randn(D_in, H, requires_grad=True)
w2 = torch.randn(H, D_out, requires_grad=True)

learning_rate = 1e-6
for t in range(500):
    y_pred = my_relu(x.mm(w1)).mm(w2)    # notre ReLU à la place de clamp()
    loss = (y_pred - y).pow(2).sum()

    if t % 100 == 99:
        print(t, loss.item())

    loss.backward()                      # appelle MyReLU.backward

    with torch.no_grad():
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad
        w1.grad.zero_()
        w2.grad.zero_()

## 2.4 PyTorch : `nn` (page 43)

- **Création du modèle** : une séquence de couches ; chaque couche est un objet avec des paramètres à apprendre ;
- **Propagation** : prédictions et coût ; `nn.functional` fournit les fonctions de coût ;
- **Rétro-propagation** automatique ;
- **Descente du gradient** sur tous les paramètres du modèle.

In [ ]:
import torch

N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

# Création du modèle : suite de couches (chaque Linear contient poids ET biais)
model = torch.nn.Sequential(
          torch.nn.Linear(D_in, H),
          torch.nn.ReLU(),
          torch.nn.Linear(H, D_out))

learning_rate = 1e-2
for t in range(500):
    # Propagation : on appelle le modèle comme une fonction
    y_pred = model(x)
    loss = torch.nn.functional.mse_loss(y_pred, y)   # erreur quadratique moyenne

    if t % 100 == 99:
        print(t, loss.item())

    # Rétro-propagation
    loss.backward()

    # Descente du gradient sur chaque paramètre du modèle
    with torch.no_grad():
        for param in model.parameters():
            param -= learning_rate * param.grad
    model.zero_grad()   # remise à zéro des gradients de tous les paramètres

## 2.5 PyTorch : optimizer (page 44)

- **Créer un optimizer** : il définit les règles de mise à jour des paramètres (ici **Adam**) ;
- `optimizer.step()` + `optimizer.zero_grad()` = **une itération de descente du gradient**.

In [ ]:
import torch

N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

model = torch.nn.Sequential(
          torch.nn.Linear(D_in, H),
          torch.nn.ReLU(),
          torch.nn.Linear(H, D_out))

learning_rate = 1e-4
# L'optimizer reçoit les paramètres à optimiser et le pas d'apprentissage
optimizer = torch.optim.Adam(model.parameters(),
                             lr=learning_rate)
for t in range(500):
    y_pred = model(x)
    loss = torch.nn.functional.mse_loss(y_pred, y)

    if t % 100 == 99:
        print(t, loss.item())

    loss.backward()

    # Une itération de descente du gradient
    optimizer.step()        # mise à jour des paramètres à partir de leurs .grad
    optimizer.zero_grad()   # remise à zéro des gradients

## 2.6 PyTorch : new class (page 45)

Définir ses propres modules `nn`, composés de couches ou d'autres modules :

- `__init__` : **initialisation** (création des couches) ;
- `forward` : **fonction de propagation** utilisant les modules fils ;
- la **rétro-propagation** est effectuée par défaut par autograd.

In [ ]:
import torch

class TwoLayerNet(torch.nn.Module):
    def __init__(self, D_in, H, D_out):
        super(TwoLayerNet, self).__init__()
        # Les couches rangées dans self deviennent des paramètres du modèle
        self.linear1 = torch.nn.Linear(D_in, H)
        self.linear2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        h_relu = self.linear1(x).clamp(min=0)   # linéaire + ReLU
        y_pred = self.linear2(h_relu)           # linéaire
        return y_pred

N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

# Construction et apprentissage d'une instance du modèle
model = TwoLayerNet(D_in, H, D_out)

optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)
for t in range(500):
    y_pred = model(x)
    loss = torch.nn.functional.mse_loss(y_pred, y)

    if t % 100 == 99:
        print(t, loss.item())

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

## 2.7 PyTorch : new modules (page 46)

On peut **mélanger des modules personnalisés et des modèles séquentiels**.
`ParallelBlock` applique deux couches linéaires en parallèle sur la même entrée, multiplie leurs
sorties élément par élément, puis applique une ReLU. Le modèle empile plusieurs instances de ce bloc.

In [ ]:
import torch

# Définition d'un nouveau module
class ParallelBlock(torch.nn.Module):
    def __init__(self, D_in, D_out):
        super(ParallelBlock, self).__init__()
        self.linear1 = torch.nn.Linear(D_in, D_out)
        self.linear2 = torch.nn.Linear(D_in, D_out)
    def forward(self, x):
        h1 = self.linear1(x)            # branche 1
        h2 = self.linear2(x)            # branche 2
        return (h1 * h2).clamp(min=0)   # produit élément par élément puis ReLU

N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

# Définition d'un modèle empilant plusieurs instances du nouveau module
model = torch.nn.Sequential(
            ParallelBlock(D_in, H),
            ParallelBlock(H, H),
            torch.nn.Linear(H, D_out))

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
for t in range(500):
    y_pred = model(x)
    loss = torch.nn.functional.mse_loss(y_pred, y)

    if t % 100 == 99:
        print(t, loss.item())

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

---
# Partie 3 : exemple de code PyTorch complet, FashionMNIST (pages 47 à 51)

FashionMNIST : 70 000 images de vêtements en niveaux de gris (28 × 28 pixels), 10 classes.

## 3.1 Chargement des données (page 47)

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda
import matplotlib.pyplot as plt

# Taille des lots (non définie sur la diapositive ; 64 est la valeur du tutoriel PyTorch)
batch_size = 64

# Download training data from open datasets.  (60 000 images d'entraînement)
training_data = datasets.FashionMNIST(
    root="data",           # dossier de stockage
    train=True,            # partie entraînement
    download=True,         # téléchargement si absent
    transform=ToTensor(),  # image -> tenseur, pixels ramenés dans [0, 1]
)

# Download test data from open datasets.  (10 000 images de test)
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

# Create data loaders.  (découpage en lots de batch_size images)
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

*Ajout :* visualisation d'une image par classe, comme la grille de la diapositive.

In [ ]:
labels = ["T_shirt", "Trouser", "Pullover", "Dress", "Coat",
          "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for k in range(10):
    i = (training_data.targets == k).nonzero()[0].item()   # première image de la classe k
    img, label = training_data[i]
    axes[k].imshow(img.squeeze(), cmap="gray_r")           # 1 x 28 x 28 -> 28 x 28
    axes[k].set_title(labels[label], fontsize=9)
    axes[k].axis("off")
plt.show()

## 3.2 Définition du modèle (page 48)

In [ ]:
# Get cpu or gpu device for training.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()              # image 1 x 28 x 28 -> vecteur de 784
        self.linear_relu_stack = nn.Sequential(  # perceptron 784 -> 512 -> 512 -> 10
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)                   # un score (logit) par classe
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)   # création du modèle et envoi sur le device
print(model)

## 3.3 Fonction d'entraînement (page 49)

In [ ]:
# Entropie croisée : classification multi-classes (applique elle-même le softmax)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)


def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)   # nombre total d'images
    model.train()                    # mode entraînement
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Affichage de la perte tous les 100 lots
        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)   # current : images déjà vues
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

## 3.4 Fonction de test et boucle sur les époques (page 50)

Une **époque** = un passage complet sur les 60 000 images d'entraînement.
Le cours obtient environ 49,6 % de bonnes réponses après la première époque.

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)   # nombre d'images de test
    num_batches = len(dataloader)    # nombre de lots
    model.eval()                     # mode évaluation
    test_loss, correct = 0, 0
    with torch.no_grad():            # pas de gradient pendant l'évaluation
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            # classe prédite = indice du plus grand logit ; on compte les bonnes réponses
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches   # perte moyenne par lot
    correct /= size            # proportion de bonnes réponses
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

## 3.5 Sauvegarde, rechargement et prédiction (page 51)

In [ ]:
# Sauvegarde des paramètres appris (state_dict : dictionnaire nom -> tenseur)
torch.save(model.state_dict(), "model.pth")

In [ ]:
# Nouveau modèle dans lequel on recharge les poids sauvegardés
model = NeuralNetwork()
model.load_state_dict(torch.load("model.pth"))

classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]   # première image de test et sa vraie classe
with torch.no_grad():
    pred = model(x)
    # pred[0] : les 10 scores ; argmax = indice de la classe la plus probable
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

---
# Partie 4 : modèles pré-entraînés (page 52)

`torchvision.models` donne accès à des réseaux déjà entraînés sur ImageNet.

- Le code du cours utilise `pretrained=True`, remplacé depuis torchvision 0.13 par `weights="DEFAULT"`.
- Les trois modèles pèsent environ **900 Mo** au total. *Ajout :* le téléchargement ne se fait que si
  `TELECHARGER = True`.

In [ ]:
import torch
import torchvision

TELECHARGER = False   # passer à True pour télécharger les poids (~900 Mo)

if TELECHARGER:
    # Code du cours (ancienne syntaxe) :
    #   alexnet = torchvision.models.alexnet(pretrained=True)
    #   vgg16 = torchvision.models.vgg16(pretrained=True)
    #   resnet101 = torchvision.models.resnet101(pretrained=True)
    # Syntaxe actuelle équivalente :
    alexnet = torchvision.models.alexnet(weights="DEFAULT")
    vgg16 = torchvision.models.vgg16(weights="DEFAULT")
    resnet101 = torchvision.models.resnet101(weights="DEFAULT")
    print(resnet101)
else:
    print("Téléchargement désactivé (TELECHARGER = False).")

---
# Partie 5 : TensorFlow, TensorBoard et Keras (pages 53 à 60)

## 5.1 TensorFlow avant 2.0 (page 53)

Deux temps : **définir le graphe de calcul**, puis **l'exécuter** (éventuellement plusieurs fois).
Ce code est détaillé pas à pas dans la partie 7 (pages 67 à 77).

In [ ]:
import numpy as np
import tensorflow as tf
tf1 = tf.compat.v1

with tf.Graph().as_default():
    # ---- Définition du graphe de calcul ----
    N, D, H = 64, 1000, 100
    x = tf1.placeholder(tf.float32, shape=(N, D))
    y = tf1.placeholder(tf.float32, shape=(N, D))
    w1 = tf1.placeholder(tf.float32, shape=(D, H))
    w2 = tf1.placeholder(tf.float32, shape=(H, D))

    h = tf.maximum(tf.matmul(x, w1), 0)                            # couche cachée + ReLU
    y_pred = tf.matmul(h, w2)                                      # sortie
    diff = y_pred - y
    loss = tf.reduce_mean(tf.reduce_sum(diff ** 2, axis=1))        # coût L2 moyen

    grad_w1, grad_w2 = tf1.gradients(loss, [w1, w2])

    # ---- Exécution du graphe de calcul ----
    with tf1.Session() as sess:
        values = {x: np.random.randn(N, D),
                  w1: np.random.randn(D, H),
                  w2: np.random.randn(H, D),
                  y: np.random.randn(N, D),}
        out = sess.run([loss, grad_w1, grad_w2],
                       feed_dict=values)
        loss_val, grad_w1_val, grad_w2_val = out

print("loss =", loss_val)

## 5.2 TensorFlow après 2.0 (page 54)

Plus de placeholder ni de session : les calculs s'exécutent immédiatement (mode *eager*).

- les données sont des tenseurs ; les **poids** sont des `tf.Variable` ;
- `tf.GradientTape` enregistre les opérations de la passe avant ;
- `tape.gradient` calcule les gradients, `optimizer.apply_gradients` met les poids à jour.

In [ ]:
import numpy as np
import tensorflow as tf

N, D, H = 64, 1000, 100

x = tf.convert_to_tensor(np.random.randn(N, D), np.float32)
y = tf.convert_to_tensor(np.random.randn(N, D), np.float32)
w1 = tf.Variable(tf.random.uniform((D, H)))   # weights
w2 = tf.Variable(tf.random.uniform((H, D)))   # weights

optimizer = tf.optimizers.SGD(1e-6)

for t in range(50):
    # La "bande" enregistre la passe avant pour pouvoir la dériver
    with tf.GradientTape() as tape:
        h = tf.maximum(tf.matmul(x, w1), 0)
        y_pred = tf.matmul(h, w2)
        diff = y_pred - y
        loss = tf.losses.MeanSquaredError()(y_pred, y)
    gradients = tape.gradient(loss, [w1, w2])
    # zip associe chaque gradient à sa variable : [(grad_w1, w1), (grad_w2, w2)]
    optimizer.apply_gradients(zip(gradients, [w1, w2]))

    if t % 10 == 9:
        print(t, float(loss))

## 5.3 TensorBoard (pages 55, 56, 59 et 60)

TensorBoard affiche des graphiques à partir de **journaux** (*logs*) écrits pendant l'apprentissage :
courbe de la perte, distribution des poids `w1`, `w2`, et graphe de calcul du modèle.

*Ajout (les diapositives ne montrent que le résultat) :* la même boucle qu'en 5.2, qui enregistre la perte
et les histogrammes des poids dans le dossier `logs/`.

Pour afficher le tableau de bord, lancer dans un terminal : `tensorboard --logdir logs`
(puis ouvrir http://localhost:6006), ou dans une cellule Jupyter : `%load_ext tensorboard` puis `%tensorboard --logdir logs`.

In [ ]:
import numpy as np
import tensorflow as tf

N, D, H = 64, 1000, 100
x = tf.convert_to_tensor(np.random.randn(N, D), np.float32)
y = tf.convert_to_tensor(np.random.randn(N, D), np.float32)
w1 = tf.Variable(tf.random.uniform((D, H)))
w2 = tf.Variable(tf.random.uniform((H, D)))
optimizer = tf.optimizers.SGD(1e-6)

# Écrivain de journaux TensorBoard
writer = tf.summary.create_file_writer("logs")

for t in range(100):
    with tf.GradientTape() as tape:
        h = tf.maximum(tf.matmul(x, w1), 0)
        y_pred = tf.matmul(h, w2)
        loss = tf.losses.MeanSquaredError()(y_pred, y)
    gradients = tape.gradient(loss, [w1, w2])
    optimizer.apply_gradients(zip(gradients, [w1, w2]))

    # Enregistrement de la perte et de la distribution des poids à l'étape t
    with writer.as_default():
        tf.summary.scalar("loss", loss, step=t)
        tf.summary.histogram("w1", w1, step=t)
        tf.summary.histogram("w2", w2, step=t)

writer.flush()
print("Journaux écrits dans le dossier logs/")

## 5.4 Keras (pages 57 et 58)

Keras est une **librairie de haut niveau**, intégrée à TensorFlow, pour faire des modèles courants facilement :

1. **définir un modèle** comme une séquence de couches ;
2. **définir un optimizer** ;
3. **construire le modèle** et définir la fonction de coût (`compile`) ;
4. **apprentissage du modèle** (`fit`).

Le code de la diapositive utilise l'ancienne API de Keras, qui ne fonctionne plus :

```python
from keras.models import Sequential
from keras.layers.core import Dense, Activation
from keras.optimizers import SGD

model = Sequential()
model.add(Dense(input_dim=D, output_dim=H))
model.add(Activation('relu'))
model.add(Dense(input_dim=H, output_dim=D))
optimizer = SGD(lr=1e0)
model.compile(loss='mean_squared_error', optimizer=optimizer)
history = model.fit(x, y, nb_epoch=50, batch_size=N, verbose=0)
```

Correspondances avec la version actuelle : `output_dim` → `units`, `input_dim` → une couche `Input`,
`lr` → `learning_rate`, `nb_epoch` → `epochs`.

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Activation
from tensorflow.keras.optimizers import SGD
import matplotlib.pyplot as plt

N, D, H = 64, 1000, 100

# 1. Modèle = séquence de couches
model = Sequential()
model.add(Input(shape=(D,)))        # dimension d'entrée (ancien input_dim=D)
model.add(Dense(H))                 # couche dense D -> H
model.add(Activation('relu'))
model.add(Dense(D))                 # couche dense H -> D

# 2. Optimizer  (pas de 1 comme sur la diapositive ; si la perte devient nan, essayer 1e-2)
optimizer = SGD(learning_rate=1e0)

# 3. Construction du modèle et fonction de coût
model.compile(loss='mean_squared_error',
              optimizer=optimizer)

# 4. Apprentissage
x = np.random.randn(N, D)
y = np.random.randn(N, D)
history = model.fit(x, y, epochs=50,
                    batch_size=N, verbose=0)

# Ajout : courbe de la perte au fil des époques
plt.plot(history.history["loss"])
plt.xlabel("epoch"); plt.ylabel("loss")
plt.show()

---
# Partie 6 : calcul distribué et choix du framework (pages 61 à 64)

*Ces diapositives ne contiennent pas de code ; en voici le résumé.*

**Modèle distribué vs données distribuées (page 61).** Pour calculer sur plusieurs GPU :

- **modèle distribué** : chaque GPU porte une partie du réseau (des couches différentes) ;
- **données distribuées** : chaque GPU porte une copie complète du modèle et traite une partie du mini-batch.

**PyTorch vs TensorFlow (pages 62 à 64).**

- En recherche, PyTorch est devenu majoritaire dans les publications des grandes conférences (2017-2019).
- En entreprise, aucune tendance nette ; TensorFlow domine dans l'embarqué et le mobile.
- Recommandation de Fei-Fei Li : **PyTorch** pour son API claire et ses graphes dynamiques (faciles à
  déboguer) ; **TensorFlow** est devenu plus intuitif depuis la 2.0, avec une grande communauté,
  plutôt via une surcouche comme Keras.

---
# Partie 7 : TensorFlow 1.x pas à pas (pages 66 à 77)

On reprend le réseau à deux couches (coût L2, descente du gradient, données aléatoires) et on
l'améliore étape par étape. Tous les codes sont dans un graphe local `with tf.Graph().as_default():`.

In [ ]:
# Imports communs à toute la partie 7
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
tf1 = tf.compat.v1   # API TensorFlow 1.x

## 7.1 Construction et exécution du graphe (pages 67 à 69)

- **Création des placeholders** : entrée `x`, sortie `y`, paramètres `W1`, `W2` ;
- **Phase de propagation** : prédictions et fonction de coût ;
- **Gradient** de la fonction de coût par rapport à `W1` et `W2` ;
- `values` : **tableaux NumPy** (entrées, sorties, paramètres) envoyés aux placeholders ;
- **Exécution** : `sess.run` calcule `loss`, `grad_w1`, `grad_w2`, récupérés sous forme de tableaux NumPy.

In [ ]:
with tf.Graph().as_default():
    # ---- Construction du graphe de calcul ----
    N, D, H = 64, 1000, 100
    x = tf1.placeholder(tf.float32, shape=(N, D))    # entrée
    y = tf1.placeholder(tf.float32, shape=(N, D))    # sortie
    w1 = tf1.placeholder(tf.float32, shape=(D, H))   # paramètres
    w2 = tf1.placeholder(tf.float32, shape=(H, D))

    h = tf.maximum(tf.matmul(x, w1), 0)
    y_pred = tf.matmul(h, w2)
    diff = y_pred - y
    loss = tf.reduce_mean(tf.reduce_sum(diff ** 2, axis=1))

    grad_w1, grad_w2 = tf1.gradients(loss, [w1, w2])

    # ---- Exécution du graphe de calcul ----
    with tf1.Session() as sess:
        values = {x: np.random.randn(N, D),
                  w1: np.random.randn(D, H),
                  w2: np.random.randn(H, D),
                  y: np.random.randn(N, D),}
        out = sess.run([loss, grad_w1, grad_w2],
                       feed_dict=values)
        loss_val, grad_w1_val, grad_w2_val = out

print("loss =", loss_val)
print("forme de grad_w1 :", grad_w1_val.shape)

## 7.2 Apprentissage : exécutions multiples du graphe (pages 70 et 71)

On exécute le graphe 50 fois et on met à jour les poids avec les gradients.

**Problème (page 71) :** les poids sont des tableaux NumPy (sur le **CPU**) : à chaque itération, ils sont
copiés vers TensorFlow (éventuellement sur le **GPU**) puis les gradients reviennent. Ces copies ralentissent tout.

*Ajout :* on garde les valeurs de la perte pour tracer la courbe « fonction de coût vs epoch ».

In [ ]:
with tf.Graph().as_default():
    N, D, H = 64, 1000, 100
    x = tf1.placeholder(tf.float32, shape=(N, D))
    y = tf1.placeholder(tf.float32, shape=(N, D))
    w1 = tf1.placeholder(tf.float32, shape=(D, H))
    w2 = tf1.placeholder(tf.float32, shape=(H, D))

    h = tf.maximum(tf.matmul(x, w1), 0)
    y_pred = tf.matmul(h, w2)
    diff = y_pred - y
    loss = tf.reduce_mean(tf.reduce_sum(diff ** 2, axis=1))
    grad_w1, grad_w2 = tf1.gradients(loss, [w1, w2])

    with tf1.Session() as sess:
        values = {x: np.random.randn(N, D),
                  w1: np.random.randn(D, H),
                  w2: np.random.randn(H, D),
                  y: np.random.randn(N, D),}
        learning_rate = 1e-5
        losses = []
        for t in range(50):
            out = sess.run([loss, grad_w1, grad_w2],
                           feed_dict=values)
            loss_val, grad_w1_val, grad_w2_val = out
            # Mise à jour des poids côté NumPy (copie CPU <-> TensorFlow à chaque tour)
            values[w1] -= learning_rate * grad_w1_val
            values[w2] -= learning_rate * grad_w2_val
            losses.append(loss_val)

plt.plot(losses, "o")
plt.title("Fonction de coût vs epoch"); plt.xlabel("epoch")
plt.show()

## 7.3 Poids en `Variable`… mais la perte ne diminue pas (pages 72 et 73)

- `W1` et `W2` ne sont plus des placeholders (à fournir à chaque exécution) mais des **Variables**,
  qui **persistent dans le graphe** de calcul : plus de copie CPU ↔ GPU.
- La **mise à jour** est décrite dans le graphe avec `assign`.
- `global_variables_initializer()` : **initialisation unique** de `W1`, `W2`.

**Problème (page 73) :** la fonction de coût ne diminue pas. On ne demande à `sess.run` que `loss` ;
TensorFlow n'exécute que les noeuds nécessaires pour la calculer, donc **les noeuds `new_w1`, `new_w2`
ne sont jamais exécutés** et les poids ne changent pas.

In [ ]:
with tf.Graph().as_default():
    N, D, H = 64, 1000, 100
    x = tf1.placeholder(tf.float32, shape=(N, D))
    y = tf1.placeholder(tf.float32, shape=(N, D))
    # Poids = Variables : ils vivent dans le graphe
    w1 = tf1.Variable(tf1.random_normal((D, H)))
    w2 = tf1.Variable(tf1.random_normal((H, D)))

    h = tf.maximum(tf.matmul(x, w1), 0)
    y_pred = tf.matmul(h, w2)
    diff = y_pred - y
    loss = tf.reduce_mean(tf.reduce_sum(diff ** 2, axis=1))
    grad_w1, grad_w2 = tf1.gradients(loss, [w1, w2])

    # Mise à jour des poids décrite DANS le graphe
    learning_rate = 1e-5
    new_w1 = w1.assign(w1 - learning_rate * grad_w1)
    new_w2 = w2.assign(w2 - learning_rate * grad_w2)

    with tf1.Session() as sess:
        sess.run(tf1.global_variables_initializer())   # initialisation unique
        values = {x: np.random.randn(N, D),
                  y: np.random.randn(N, D),}
        losses = []
        for t in range(50):
            # On ne demande que loss : new_w1 / new_w2 ne sont pas exécutés !
            loss_val, = sess.run([loss], feed_dict=values)
            losses.append(loss_val)

plt.plot(losses, "o")
plt.title("La perte reste constante"); plt.xlabel("epoch")
plt.show()

## 7.4 Solution : un noeud `updates` (page 74)

On **ajoute au graphe un noeud** qui regroupe le calcul des nouveaux poids (`tf.group`), puis on
**demande le calcul de ce noeud** à chaque `sess.run`.

In [ ]:
with tf.Graph().as_default():
    N, D, H = 64, 1000, 100
    x = tf1.placeholder(tf.float32, shape=(N, D))
    y = tf1.placeholder(tf.float32, shape=(N, D))
    w1 = tf1.Variable(tf1.random_normal((D, H)))
    w2 = tf1.Variable(tf1.random_normal((H, D)))

    h = tf.maximum(tf.matmul(x, w1), 0)
    y_pred = tf.matmul(h, w2)
    diff = y_pred - y
    loss = tf.reduce_mean(tf.reduce_sum(diff ** 2, axis=1))
    grad_w1, grad_w2 = tf1.gradients(loss, [w1, w2])

    learning_rate = 1e-5
    new_w1 = w1.assign(w1 - learning_rate * grad_w1)
    new_w2 = w2.assign(w2 - learning_rate * grad_w2)
    updates = tf.group(new_w1, new_w2)   # noeud qui exécute les deux mises à jour

    with tf1.Session() as sess:
        sess.run(tf1.global_variables_initializer())
        values = {x: np.random.randn(N, D),
                  y: np.random.randn(N, D),}
        losses = []
        for t in range(50):
            # On demande loss ET updates : les poids sont mis à jour à chaque tour
            loss_val, _ = sess.run([loss, updates],
                                   feed_dict=values)
            losses.append(loss_val)

plt.plot(losses, "o")
plt.title("La perte diminue"); plt.xlabel("epoch")
plt.show()

## 7.5 Optimizer (page 75)

Un **optimizer** calcule les gradients et met les poids à jour : `optimizer.minimize(loss)` remplace
`tf.gradients` + `assign` + `tf.group`. On **appelle l'optimizer dans la session** via `updates`.

In [ ]:
with tf.Graph().as_default():
    N, D, H = 64, 1000, 100
    x = tf1.placeholder(tf.float32, shape=(N, D))
    y = tf1.placeholder(tf.float32, shape=(N, D))
    w1 = tf1.Variable(tf1.random_normal((D, H)))
    w2 = tf1.Variable(tf1.random_normal((H, D)))

    h = tf.maximum(tf.matmul(x, w1), 0)
    y_pred = tf.matmul(h, w2)
    diff = y_pred - y
    loss = tf.reduce_mean(tf.reduce_sum(diff * diff, axis=1))

    # L'optimizer ajoute au graphe le calcul des gradients ET la mise à jour des poids
    optimizer = tf1.train.GradientDescentOptimizer(1e-5)
    updates = optimizer.minimize(loss)

    with tf1.Session() as sess:
        sess.run(tf1.global_variables_initializer())
        values = {x: np.random.randn(N, D),
                  y: np.random.randn(N, D),}
        losses = []
        for t in range(50):
            loss_val, _ = sess.run([loss, updates],
                                   feed_dict=values)
            losses.append(loss_val)

print("perte initiale :", losses[0], "- perte finale :", losses[-1])

## 7.6 Fonction de coût prédéfinie (page 76)

`tf.losses.mean_squared_error` remplace le calcul écrit à la main.
Attention : c'est une **moyenne** sur tous les éléments (et non une somme par ligne), la perte est donc
beaucoup plus petite ; d'où un pas d'apprentissage plus grand (`1e-3`).

In [ ]:
with tf.Graph().as_default():
    N, D, H = 64, 1000, 100
    x = tf1.placeholder(tf.float32, shape=(N, D))
    y = tf1.placeholder(tf.float32, shape=(N, D))
    w1 = tf1.Variable(tf1.random_normal((D, H)))
    w2 = tf1.Variable(tf1.random_normal((H, D)))

    h = tf.maximum(tf.matmul(x, w1), 0)
    y_pred = tf.matmul(h, w2)
    # Fonction de coût prédéfinie (l'ordre des arguments n'a pas d'effet : l'erreur est au carré)
    loss = tf1.losses.mean_squared_error(y_pred, y)

    optimizer = tf1.train.GradientDescentOptimizer(1e-3)
    updates = optimizer.minimize(loss)

    with tf1.Session() as sess:
        sess.run(tf1.global_variables_initializer())
        values = {x: np.random.randn(N, D),
                  y: np.random.randn(N, D),}
        losses = []
        for t in range(50):
            loss_val, _ = sess.run([loss, updates],
                                   feed_dict=values)
            losses.append(loss_val)

print("perte initiale :", losses[0], "- perte finale :", losses[-1])

## 7.7 Couches et initialisation Xavier (page 77)

- `tf.layers.dense` **met en place automatiquement les paramètres de la couche** (poids et biais) ;
- **initialisation Xavier** (aussi appelée *Glorot*) : poids initiaux d'échelle adaptée à la taille des couches.

Adaptations :

- `tf.contrib` n'existe plus en TF 2 : `tf.contrib.layers.xavier_initializer()` devient `tf1.glorot_uniform_initializer()` ;
- `tf.layers.dense` n'est plus disponible avec Keras 3 (TensorFlow ≥ 2.16) : on importe la même fonction
  depuis le paquet `tf-keras` (ancien Keras), installé avec `pip install tf-keras`.

In [ ]:
# Ancienne fonction tf.layers.dense, fournie par le paquet tf-keras
from tf_keras.src.legacy_tf_layers.core import dense

with tf.Graph().as_default():
    N, D, H = 64, 1000, 100
    x = tf1.placeholder(tf.float32, shape=(N, D))
    y = tf1.placeholder(tf.float32, shape=(N, D))

    # Initialisation Xavier / Glorot (ex tf.contrib.layers.xavier_initializer)
    init = tf1.glorot_uniform_initializer()
    # Couches denses : poids et biais créés automatiquement
    h = dense(inputs=x, units=H,
            activation=tf.nn.relu, kernel_initializer=init)
    y_pred = dense(inputs=h, units=D,
            kernel_initializer=init)

    loss = tf1.losses.mean_squared_error(y_pred, y)

    optimizer = tf1.train.GradientDescentOptimizer(1e0)
    updates = optimizer.minimize(loss)

    with tf1.Session() as sess:
        sess.run(tf1.global_variables_initializer())
        values = {x: np.random.randn(N, D),
                  y: np.random.randn(N, D),}
        losses = []
        for t in range(50):
            loss_val, _ = sess.run([loss, updates],
                                   feed_dict=values)
            losses.append(loss_val)

print("perte initiale :", losses[0], "- perte finale :", losses[-1])

---
# Bilan

| | NumPy | TensorFlow 1.x | TensorFlow 2.x / Keras | PyTorch |
|---|---|---|---|---|
| Graphe | aucun | statique : construit puis exécuté (`Session`) | dynamique (*eager*) | dynamique (construit à la volée) |
| Gradients | à la main | `tf.gradients` / `optimizer.minimize` | `GradientTape` / `fit` | `loss.backward()` |
| Poids | tableaux NumPy | `placeholder` puis `Variable` + `assign` | `tf.Variable`, couches `Dense` | `requires_grad=True`, `nn.Linear` |
| Mise à jour | à la main | noeud `updates` exécuté par `sess.run` | `apply_gradients` / `fit` | `optimizer.step()` |
| GPU | non | `tf.device('/gpu:0')` | automatique | `.cuda()` / `.to(device)` |

**Idée centrale :** les frameworks construisent le graphe de calcul et automatisent la rétro-propagation ;
il suffit de décrire la passe avant. Les couches (`nn`, `layers`), les fonctions de coût et les
optimiseurs prédéfinis simplifient ensuite l'écriture du modèle et de son apprentissage.